In [1]:
import import_ipynb
import json
import re
import asyncio
from typing import List, Dict, Optional, Any
from pydantic import BaseModel, Field
from ollama import AsyncClient
from SPARQLWrapper import SPARQLWrapper, JSON
from context_pack_core import * ; # For now keep this, once context_pack.ipynb is converted into module, can directly import from that

In [2]:
def build_extractor_prompt_template(context_string: str) -> str:
    return f"""You are an entity mention extractor for a knowledge graph.

Goal
Extract entity mentions from the user query and select the correct STRING-LABEL predicate
to later retrieve matching entities from the knowledge graph.

Hard rules
1.  **Based on User Text:** Output must be based on contiguous substrings of the user text.
2.  **Trim Noise (CRITICAL):**
    -   For **Person** names: ALWAYS remove citation suffixes like "et al", "al.", "ibid", or reference numbers (e.g. "[1]") from the extracted text. Return ONLY the name (e.g., extract "Vaswani" NOT "Vaswani et al").
    -   For **Titles**: Keep punctuation if it is part of the title.
3.  **Atomic Mentions (CRITICAL):**
    -   Do NOT extract a mention if it is a substring of another extracted mention.
    -   Example: If the text contains "... for the DBLP Scholarly Knowledge Graph", extract the full Title. Do NOT extract "DBLP" separately as a Venue/Repository if it is just part of that title.
    -   Only extract the longest, most specific entity concept.
4.  **No Hallucinations:** Do not infer or complete missing information.
5.  **Output Format:** Return ONLY valid JSON.

How to choose `type`
- `type` MUST be exactly one of the Types listed in Schema Context.
- Output the TYPE CURIE exactly as shown (the left of the "|"), NOT the label text.

How to choose `label_pred` (CRITICAL)
- `label_pred` MUST be exactly one predicate CURIE listed under the chosen `type`.
- IMPORTANT: In Schema Context predicate rows have the shape:
    pred_curie | pred_label | rng:...
  You MUST output the pred_curie (left side). Ignore pred_label entirely.
- `label_pred` MUST have rng:lit(xsd:string).
- Choose the predicate whose VALUES are most likely to contain the mention text verbatim.
- If multiple predicates are plausible, pick the best one.

Attrs
- attrs are optional metadata, NOT part of the mention text.
- Only use attribute keys that appear in Schema Context for that type.
- Values MUST be verbatim substrings of the user query.
- Do not emit empty strings.
- Year ranges:
  If a range appears (e.g., "2015-2020" or "between 2015 and 2020"),
  store it on ONE mention as "year_start":"2015","year_end":"2020".

Schema Context
{context_string}

Output JSON schema:
{{"mentions":[{{ "text":str, "type":str, "label_pred":str, "attrs":{{str:str}} }}]}}
Return only JSON, no prose.
"""

<!-- Vaswani dblp:Pub lication
dblp:Person


query kg using sparql - 0 result


the type of vaswani is rdf -->

In [3]:
from typing import List, Dict, Optional, Any
from pydantic import BaseModel, Field

# SPARQL Helpers
def run_sparql(endpoint: str, query: str, timeout: int = 15) -> List[Dict]:
    """Execute SPARQL query against endpoint with robust error handling."""
    s = SPARQLWrapper(endpoint)
    s.setMethod("POST")
    s.setTimeout(timeout)
    s.setReturnFormat(JSON)
    s.setQuery(query)
    try:
        ret = s.query().convert()
        return ret.get("results", {}).get("bindings", [])
    except Exception as e:
        # print(f"SPARQL error: {e}") 
        return []

def extract_value(binding: dict, var: str) -> Optional[str]:
    if var in binding:
        return binding[var].get("value")
    return None

def get_full_iri(curie: str, schema: SchemaIndex) -> str:
    """Expands a CURIE to a full IRI using the schema namespaces."""
    if curie.startswith("http://") or curie.startswith("https://"):
        return curie
    if ":" not in curie:
        return curie
    prefix, local = curie.split(":", 1)
    if prefix in schema.namespaces:
        return schema.namespaces[prefix] + local
    return curie

# Models
class Mention(BaseModel):
    text: str
    type: str
    label_pred: Optional[str] = Field(None, description="The predicate label that indicates the type of this mention")
    attrs: Dict[str, str] = Field(default_factory=dict)
    
    # Logprob metadata
    type_logprob: Optional[float] = None
    pred_logprob: Optional[float] = None
    
    # Raw alternatives - have to keep token level - TODO: check some better ways maybe Gliner v2?
    type_alternatives: Optional[List[Dict[str, Any]]] = None
    
    # Resolved alternatives - schema level
    resolved_type_alternatives: Optional[List[str]] = Field(default_factory=list)
    
    uncertain_type_token: Optional[str] = None

    # Verification Status
    verification_status: str = "unverified" # unverified, verified, corrected, not_found, type_mismatch
    original_type: Optional[str] = None # stores original LLM guess if corrected

class MentionList(BaseModel):
    mentions: List[Mention]

def sanitize_response(content: str) -> str:
    match = re.search(r'\{.*\}', content, re.DOTALL)
    if match:
        return match.group(0)
    raise ValueError("No JSON object found in the response")

def resolve_candidate(full_str: str, uncertain_token: str, alt_token: str, schema: SchemaIndex) -> Optional[str]:
    if not uncertain_token or not alt_token:
        return None
    
    idx = full_str.find(uncertain_token)
    
    if idx == -1: return None
    
    search_prefix = full_str[:idx] + alt_token
    candidates = []
    
    # Only resolving Types here
    for iri in schema.classes.keys():
        curie = make_curie(iri, schema.namespaces)
        if curie.startswith(search_prefix):
            candidates.append(curie)
    if not candidates: return None
    candidates.sort(key=len)
    return candidates[0]

def assign_logprobs_to_mentions(content: str, logprobs: list, mentions: List[Mention], schema: SchemaIndex) -> List[Mention]:
    if not logprobs: return mentions
    token_positions = []
    curr_char_pos = 0
    for lp in logprobs:
        token = lp.token
        pos = content.find(token, curr_char_pos)
        if pos != -1:
            alts = [{'token': t.token, 'logprob': t.logprob} for t in lp.top_logprobs] if hasattr(lp, 'top_logprobs') and lp.top_logprobs else []
            token_positions.append({'start': pos, 'end': pos + len(token), 'logprob': lp.logprob, 'token_text': token, 'alternatives': alts})
            curr_char_pos = pos + len(token)
            
    search_from = 0
    for m in mentions:
        val = m.type
        if not val: continue
        quoted_val = f"\"{val}\"" # Corrected escaping for double quotes
        pos = content.find(quoted_val, search_from)
        if pos != -1:
            v_start, v_end = pos + 1, pos + len(quoted_val) - 1
            t_info = [tp for tp in token_positions if tp['start'] >= v_start and tp['end'] <= v_end]
            if t_info:
                avg_lp = sum(t['logprob'] for t in t_info) / len(t_info)
                worst = min(t_info, key=lambda x: x['logprob'])
                m.type_logprob = avg_lp
                m.uncertain_type_token = worst['token_text']
                m.type_alternatives = worst['alternatives']
                if worst['alternatives']:
                    resolved = []
                    for alt in worst['alternatives']:
                        res = resolve_candidate(val, worst['token_text'], alt['token'], schema)
                        if res and res != val and res not in resolved:
                            resolved.append(res)
                    m.resolved_type_alternatives = resolved
            search_from = pos + len(quoted_val)
    return mentions

class MentionExtractor:
    def __init__(self, ollama_host: str = "http://ollama.warhol.informatik.rwth-aachen.de", client: Optional[AsyncClient] = None, model: str = "llama3.3:70b"):
        if client:
             self.client = client
        else:
             self.client = AsyncClient(host=ollama_host, timeout=180.0)
        self.model = model

    async def extract(
        self,
        text: str,
        schema: SchemaIndex,
        endpoint_url: Optional[str] = None,
        temperature: float = 0.1,
        include_predicates: bool = True,
        verify_with_kg: bool = True
    ) -> List[Mention]:
        
        # Prepare schema and context
        filtered_schema = filter_schema(schema, keep_only_literal_or_labelable_object=True, include_predicates=include_predicates)
        context_str = schema_to_minimal_context(filtered_schema, include_prefixes=True, include_predicates=include_predicates)

        # context_str = schema_to_context_string(filtered_schema)
        # print(type(filtered_schema), type(context_str))
        
        prompt = build_extractor_prompt_template(context_str)

        # LLM Extraction
        try:
            resp = await self.client.chat(
                model=self.model,
                messages=[{"role": "system", "content": prompt}, {"role": "user", "content": text}],
                logprobs=True,
                top_logprobs=5,
                options={"temperature": temperature}
            )
            content = resp.message.content
            # print(resp.message)
            ml = MentionList.model_validate_json(sanitize_response(content))
            
            # Logprob Analysis
            if hasattr(resp, 'logprobs') and resp.logprobs:
                ml.mentions = assign_logprobs_to_mentions(content, resp.logprobs, ml.mentions, schema)
            
            # KG Verification & Correction
            if verify_with_kg and endpoint_url:
                await self.verify_and_correct(ml.mentions, schema, endpoint_url)
                
            return ml.mentions
        except Exception as e:
            print(f"Error in extract: {e}")
            return []

    async def verify_and_correct(self, mentions: List[Mention], schema: SchemaIndex, endpoint_url: str):
        """
        Verify mentions against KG and correct type if necessary.
        Uses optimized queries to avoid timeouts by leveraging indexes on common label predicates.
        """
        # Build set of valid CURIEs for quick lookup
        valid_curies = set()
        for iri in schema.classes.keys():
            valid_curies.add(make_curie(iri, schema.namespaces))

        loop = asyncio.get_event_loop()

        for m in mentions:
            safe_text = m.text.replace("\"", "\\\"") # Corrected escaping for double quotes
            
            # Prepare list of predicates to check - keep scholarly kg focussed for now
            preds_to_check = [
                "http://www.w3.org/2000/01/rdf-schema#label",
                "http://www.w3.org/2004/02/skos/core#prefLabel",
                "http://xmlns.com/foaf/0.1/name",
                "http://schema.org/name",
                "http://purl.org/dc/terms/title"
            ]
            
            # Add the predicted label predicate if valid
            if m.label_pred:
                full_pred = get_full_iri(m.label_pred, schema)
                if full_pred not in preds_to_check:
                    preds_to_check.insert(0, full_pred) # Check it first
            
            # Construct VALUES block for predicates
            values_block = " ".join([f"<{p}>" for p in preds_to_check])
            
            # Use VALUES to restrict predicates, allowing index usage
            query = f"""
            SELECT DISTINCT ?type WHERE {{
              VALUES ?p {{ {values_block} }}
              ?s ?p ?label .
              FILTER(CONTAINS(LCASE(STR(?label)), LCASE("{safe_text}")))
              ?s a ?type .
            }} LIMIT 20
            """
            
            results = await loop.run_in_executor(None, run_sparql, endpoint_url, query)
            
            if not results:
                m.verification_status = "not_found"
                continue

            found_types = set()
            for r in results:
                t_iri = extract_value(r, "type")
                if t_iri:
                    found_types.add(t_iri)
            
            # Convert KG IRI types to CURIEs for comparison
            found_curies = set()
            for t_iri in found_types:
                found_curies.add(make_curie(t_iri, schema.namespaces))
            
            # Check if original prediction is correct
            if m.type in found_curies:
                m.verification_status = "verified"
                continue
            
            # Try to find a valid alternative from Schema
            valid_found = found_curies.intersection(valid_curies)
            
            print(found_curies)

            print(valid_found)

            if valid_found:
                best_correction = None
                
                # Priority 1: Overlap with LLM alternatives
                if m.resolved_type_alternatives:
                    for alt in m.resolved_type_alternatives:
                        if alt in valid_found:
                            best_correction = alt
                            break
                
                # Priority 2: Any valid type found (shortest CURIE heuristic)
                if not best_correction:
                    best_correction = sorted(list(valid_found), key=len)[0]
                
                if best_correction:
                    m.original_type = m.type
                    m.type = best_correction
                    m.verification_status = "corrected"
            else:
                m.verification_status = "type_mismatch"

# Wrapper for backward compatibility
async def extract_mentions_with_schema(
    text: str,
    schema: SchemaIndex,
    client: Optional[AsyncClient] = None,
    model: str = "llama3.3:70b",
    temperature: float = 0.1,
    include_predicates: bool = True,
    keep_only_literal_or_labelable_object: bool = True,
    endpoint_url: Optional[str] = None
) -> List[Mention]:
    
    # model = "gpt-oss:120b" # Doesn't work reliably - TODO: Figure out why, might be better to use this instead of Llama 70b
    extractor = MentionExtractor(client=client, model=model)
    return await extractor.extract(
        text, 
        schema, 
        endpoint_url=endpoint_url,
        temperature=temperature, 
        include_predicates=include_predicates,
        verify_with_kg=bool(endpoint_url)
    )


In [4]:
import pandas as pd
from IPython.display import display, HTML

def resolve_alt_label(full_str: str, uncertain_token: str, alt_token: str, schema: SchemaIndex, kind: str = "type") -> str:
    """
    Attempts to resolve an alternative token to a human-readable label.
    (Kept for compatibility, though resolve_candidate is preferred for logic)
    """
    res = resolve_candidate(full_str, uncertain_token, alt_token, schema)
    return res if res else ""

def display_mentions(mentions: List[Mention], schema: Optional[SchemaIndex] = None):
    """
    Displays extracted mentions in table format using Pandas
    """
    if not mentions:
        print("No mentions found.")
        return

    data = []
    for m in mentions:
        # Get attributes
        t_logprob = getattr(m, 'type_logprob', None)
        t_alts = getattr(m, 'type_alternatives', None)

        # print("T: ", t_alts)
        
        p_logprob = getattr(m, 'pred_logprob', None)
        p_alts = getattr(m, 'pred_alternatives', None)

        # Format Type Alternatives
        t_alts_str = ""
        # Show resolved alternatives if available prioritizing resolved
        if hasattr(m, 'resolved_type_alternatives') and m.resolved_type_alternatives:
             t_alts_str = ", ".join(m.resolved_type_alternatives)
        elif t_alts:
            items = []
            for a in t_alts:
                # Handle both dict and object access just in case
                tok = a.get('token') if isinstance(a, dict) else a.token
                lp = a.get('logprob') if isinstance(a, dict) else a.logprob
                txt = f"{tok}({lp:.2f})"
                # print("T: ", txt)
                # items.append(txt)
            t_alts_str = ", ".join(items)
            # print(t_alts_str)

        # Format Pred Alternatives
        p_alts_str = ""
        if p_alts:
            items = []
            for a in p_alts:
                tok = a.get('token') if isinstance(a, dict) else a.token
                lp = a.get('logprob') if isinstance(a, dict) else a.logprob
                txt = f"{tok}({lp:.2f})"
                items.append(txt)
            p_alts_str = ", ".join(items)

        v_status = getattr(m, 'verification_status', 'unverified')
        orig_type = getattr(m, 'original_type', None)

        print(t_alts_str)

        data.append({
            "Text": m.text,
            "Status": v_status,
            "Type": m.type,
            "Original Type": orig_type if orig_type else "",
            "Type Conf": f"{t_logprob:.4f}" if t_logprob is not None else "N/A",
            "Type Alts": t_alts_str,
            "Predicate": m.label_pred,
            "Pred Conf": f"{p_logprob:.4f}" if p_logprob is not None else "N/A",
            "Attributes": str(m.attrs) if m.attrs else ""
        })
    
    df = pd.DataFrame(data)
    display(df)


### Test

In [5]:
# ctx_pack = load_schema("./dblp_schema.rdf", base_iri="https://dblp.org/rdf/schema#")
ctx_pack = load_schema("./nobel.rdf", base_iri="http://data.nobelprize.org/terms/")
# ctx_pack = load_schema("./golem.ttl", base_iri="https://ontology.golemlab.eu/")

In [6]:
# print(schema_to_context_string(ctx_pack))

In [7]:
# test_queries = [
#     'List co-authors of Attention Is All You Need by Vaswani et al 2017',
#     'list all papers by Yoshua Bengio presented at neurips between 2015 and 2020',
#     'who are the authors of the iclr 2021 paper "self-supervised learning is all you need"?',
#     'list all papers by Michael Jordan at nips between 1995 and 2005',
#     'who are the authors of "DBLPLink 2.0 - An Entity Linker for the DBLP Scholarly Knowledge Graph"',
#     'Find papers by Random Forest in 2019',
#     'Who is Michael Stonebraker?',
#     'papers in SIGMOD 2022',
#     'publications by Hinton after 2018',
#     'list articles from 2010 to 2015 by Andrew Ng',
#     'co-authored by Geffrey Hinton and Yoshua Bengio',
#     "authors of the paper 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding'",
#     "What are the affiliations of the creators of 'GPT-3'?",
#     "Find conferences where 'Deep Residual Learning for Image Recognition' was cited more than 100 times",
#     "Show me publications about 'Graph Neural Networks' from 2020 to 2023 in ICLR or ICML",
#     "Who edited the book 'Deep Learning' by Goodfellow et al.?",
#     'List workshops co-located with CVPR 2024',
# ]

# for q in test_queries:
#     print(f"\nProcessing: {q}")
#     mentions = await extract_mentions_with_schema(q, ctx_pack)
#     display_mentions(mentions)

In [8]:
test_queries = [
    'Who won the nobel prize in physics in 1992',
    'did Benjamin List win a nobel prize in Chemistry?',
    'what field did Karl Ziegler win the nobel in?'
]

for q in test_queries:
    print(f"\nProcessing: {q}")
    mentions = await extract_mentions_with_schema(q, ctx_pack)
    display_mentions(mentions)


Processing: Who won the nobel prize in physics in 1992
http://data.nobelprize.org/terms/Laureate
http://data.nobelprize.org/terms/Laureate


,Text,Status,Type,Original Type,Type Conf,Type Alts,Predicate,Pred Conf,Attributes
0,Nobel Prize in Physics,unverified,http://data.nobelprize.org/terms/NobelPrize,,-0.0419,http://data.nobelprize.org/terms/Laureate,rdfs:label,N/A,{'year': '1992'}
1,physics,unverified,http://data.nobelprize.org/terms/Category,,-0.0000,http://data.nobelprize.org/terms/Laureate,rdfs:label,N/A,



Processing: did Benjamin List win a nobel prize in Chemistry?

http://data.nobelprize.org/terms/Laureate


,Text,Status,Type,Original Type,Type Conf,Type Alts,Predicate,Pred Conf,Attributes
0,Benjamin List,unverified,http://data.nobelprize.org/terms/Laureate,,-0.0000,,rdfs:label,N/A,
1,Nobel Prize in Chemistry,unverified,http://data.nobelprize.org/terms/NobelPrize,,-0.0011,http://data.nobelprize.org/terms/Laureate,rdfs:label,N/A,



Processing: what field did Karl Ziegler win the nobel in?

http://data.nobelprize.org/terms/Laureate


,Text,Status,Type,Original Type,Type Conf,Type Alts,Predicate,Pred Conf,Attributes
0,Karl Ziegler,unverified,http://data.nobelprize.org/terms/Laureate,,-0.0000,,rdfs:label,N/A,
1,Nobel,unverified,http://data.nobelprize.org/terms/NobelPrize,,-0.0000,http://data.nobelprize.org/terms/Laureate,rdfs:label,N/A,


In [9]:
test_queries = [
    "Who are the characters in 'Harry Potter and the Philosopher's Stone'?",
    # "List all works where 'Harry Potter' appears as a character",
    # "Does 'Ron Weasley' appear in 'Harry Potter and the Goblet of Fire'?",
    # "Who is involved in the romantic relationship with Hermione Granger?",
    # "What narrative role does 'Voldemort' play?",
    # "List characters who play the role of 'Hero'",
    # "What relationship role does Ron play in 'romantic love between Ron and Hermione'?",
    # "What event follows 'Hermione arrives at the ball with Viktor Krum'?",
    # "List all narrative events that occur during the 'Yule Ball'",
    # "What is the duration of the 'Battle of Hogwarts'?",
    # "Show me the fabula (chronological order) of 'Harry Potter and the Deathly Hallows'",
    # "What event precedes 'Ron argues with Hermione'?",
    "Who uses the Elder Wand in the Battle of Hogwarts?",
    # "Where does the 'Quidditch World Cup match' take place?",
    # "List all narrative locations in 'Harry Potter and the Philosopher's Stone'",
    # "What object is located in the 'Chamber of Secrets'?",
    # "What is Ron's psychological state during the Yule Ball?",
    # "List features associated with the character 'Harry Potter' (e.g., bravery)",
    # "Find events where the psychological state is 'Jealousy'",
    # "What is the modal target of the syuzhet of 'Battle of Hogwarts'?",
    # "List narrative units that refer to the event 'Voldemort casts Avada Kedavra'",
    # "Which fandoms are associated with the work 'Harry Potter'?"
]

for q in test_queries:
    print(f"\nProcessing: {q}")
    mentions = await extract_mentions_with_schema(q, ctx_pack)
    display_mentions(mentions)


Processing: Who are the characters in 'Harry Potter and the Philosopher's Stone'?


/home/hell/anaconda3/envs/kglab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


http://data.nobelprize.org/terms/Laureate


,Text,Status,Type,Original Type,Type Conf,Type Alts,Predicate,Pred Conf,Attributes
0,Harry Potter and the Philosopher's Stone,unverified,http://dbpedia.org/ontology/Book,,-0.0037,http://data.nobelprize.org/terms/Laureate,http://purl.org/dc/terms/title,N/A,



Processing: Who uses the Elder Wand in the Battle of Hogwarts?

http://data.nobelprize.org/terms/Laureate


,Text,Status,Type,Original Type,Type Conf,Type Alts,Predicate,Pred Conf,Attributes
0,Elder Wand,unverified,http://dbpedia.org/ontology/MagicalObject,,-0.0031,,rdfs:label,N/A,
1,Battle of Hogwarts,unverified,http://dbpedia.org/ontology/Event,,-0.0000,http://data.nobelprize.org/terms/Laureate,rdfs:label,N/A,


In [29]:
# ctx_pack = load_schema("./golem.ttl", base_iri="https://ontology.golemlab.eu/")

# filtered_schema = filter_schema(
#     ctx_pack,
#     keep_only_literal_or_labelable_object=True,
#     include_predicates=True
# )


# context_str = schema_to_minimal_context(
#     filtered_schema,
#     include_prefixes=True,
#     include_predicates=True
# )

# text = "Who are the characters in 'Harry Potter and the Philosopher's Stone'?"

# model = "llama3.3:70b"

# prompt = build_extractor_prompt_template(context_str)
# print(prompt)

In [13]:
# resp = await client.chat(
#     model=model,
#     messages=[
#         {"role": "system", "content": prompt},
#         {"role": "user", "content": text}
#     ],
#     logprobs=True,
#     top_logprobs = 3,
#     options={"temperature": 0.1}
# )



In [14]:
# from IPython.display import display

# print(type(resp))

# print(resp.model_dump_json(indent=2))

In [17]:
from gliner import GLiNER

model = GLiNER.from_pretrained("urchade/gliner_large-v2.1")

text = """
Who are the co-authors of Vaswani et. al in Attention is all you need 2017
"""

labels = ["Person", "Date", "Award", "Publication", "Conference"]

entities = model.predict_entities(text, labels, threshold=0.3)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 9054.08it/s]
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Vaswani et. al => Person
Attention is all you need => Conference
2017 => Date
